# Test Classificatori e Valutazione della Sostenibilità — MaxViT-Tiny-512
## Real+Augmented vs Real+SyntheticPositive

Equivalente MaxViT-512 di [`12_Valutazione_Sostenibilità.ipynb`](12_Valutazione_Sostenibilità.ipynb).
Confronta le prestazioni sul test set reale tra due strategie di addestramento:

| Configurazione | Dati di Training | Notebook |
|---|---|---|
| **Real+Augmented** | Reali + augmentation tradizionale (geometrica) | `18` |
| **Real+SyntheticPositive** | Reali + campioni positivi sintetici (diffusione fine-tuned) | `19` |

**Nota:** il costo di generazione dei dati (augmentation tradizionale vs training del diffusore) non dipende
dall'architettura del classificatore a valle: questo notebook riusa gli **stessi log di sostenibilità**
prodotti dai notebook `02` (augmentation) e `03b`/`04b` (diffusori), confrontandoli con le prestazioni
ottenute qui da MaxViT-Tiny-512 invece che da ResNet-50.

In [ ]:
import os
import sys

# Controlla se il notebook sta girando su Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    BASE_PATH = '/content/drive/MyDrive/MammoDiffusion/'
else:
    print("Ambiente locale rilevato.")
    BASE_PATH = '../'

print(f"Percorso base: ", BASE_PATH)

#### Import librerie

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score,
    classification_report, precision_score, recall_score, f1_score,
)

sys.path.insert(0, os.getcwd())
from maxvit_utils import build_maxvit_model, resolve_normalization, make_dataloader, predict_probs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

#### Configurazione

In [ ]:
TRAD_AUG_EXP_DIR  = os.path.join(BASE_PATH, 'experiments', 'exp_maxvit512_real_augmented')
SYNTH_POS_EXP_DIR = os.path.join(BASE_PATH, 'experiments', 'exp_maxvit512_synth_pos')

TEST_CSV_PATH = os.path.join(BASE_PATH, 'data', 'processed', 'metadata', 'test.csv')
VAL_CSV_PATH  = os.path.join(BASE_PATH, 'data', 'processed', 'metadata', 'val.csv')

IMG_SIZE = 512
BATCH_SIZE = 8
SEED = 42
IMAGENET_MEAN = (0.5, 0.5, 0.5)
IMAGENET_STD = (0.5, 0.5, 0.5)

OUTPUT_DIR      = os.path.join(BASE_PATH, 'results', 'test_maxvit512_trad_aug_vs_synth_pos')
FIGURES_DIR     = os.path.join(OUTPUT_DIR, 'figures')
TABLES_DIR      = os.path.join(OUTPUT_DIR, 'tables')
PREDICTIONS_DIR = os.path.join(OUTPUT_DIR, 'predictions')
for d in (OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, PREDICTIONS_DIR):
    os.makedirs(d, exist_ok=True)
print("output in:", OUTPUT_DIR)

#### Creazione Validation e Test dataset

In [ ]:
df_val  = pd.read_csv(VAL_CSV_PATH).reset_index(drop=True)
df_test = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)
def fix_path(p):
    if os.path.isabs(p) or p.startswith('../') or p.startswith('./'):
        return p
    return os.path.join(BASE_PATH, p)

df_val['processed_path'] = df_val['processed_path'].apply(fix_path)
df_test['processed_path'] = df_test['processed_path'].apply(fix_path)

print("VAL :", len(df_val), " img  (malato=%d, sano=%d)" % ((df_val['cancer']==1).sum(), (df_val['cancer']==0).sum()))
print("TEST:", len(df_test), " img  (malato=%d, sano=%d)" % ((df_test['cancer']==1).sum(), (df_test['cancer']==0).sum()))

val_loader  = make_dataloader(df_val,  'processed_path', 'cancer', IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
test_loader = make_dataloader(df_test, 'processed_path', 'cancer', IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

#### Caricamento Modelli

In [ ]:
model_trad_aug = build_maxvit_model(num_classes=1, pretrained=False)
_ckpt_path = os.path.join(TRAD_AUG_EXP_DIR, 'real_augmented_maxvit512_final_best.pt')
assert os.path.isfile(_ckpt_path), (
    f"Modello 'Real+Augmented' non trovato in {_ckpt_path}. "
    f"Esegui prima il notebook di training corrispondente."
)
model_trad_aug.load_state_dict(torch.load(_ckpt_path, map_location=DEVICE))
model_trad_aug.to(DEVICE)
print(f"[Real+Augmented] Modello caricato da: {_ckpt_path}")

In [ ]:
model_synth_pos = build_maxvit_model(num_classes=1, pretrained=False)
_ckpt_path = os.path.join(SYNTH_POS_EXP_DIR, 'synth_pos_maxvit512_final_best.pt')
assert os.path.isfile(_ckpt_path), (
    f"Modello 'Real+SyntheticPositive' non trovato in {_ckpt_path}. "
    f"Esegui prima il notebook di training corrispondente."
)
model_synth_pos.load_state_dict(torch.load(_ckpt_path, map_location=DEVICE))
model_synth_pos.to(DEVICE)
print(f"[Real+SyntheticPositive] Modello caricato da: {_ckpt_path}")

In [ ]:
modelli = {'Real+Augmented': model_trad_aug, 'Real+SyntheticPositive': model_synth_pos}
MODELLI_ATTIVI = list(modelli.keys())

#### Soglia di Youden (calcolata sul Validation Set)

In [ ]:
soglie = {}
val_true = None
for nome, modello in modelli.items():
    yt, yp = predict_probs(modello, val_loader, DEVICE)
    val_true = yt
    soglie[nome] = optimal_threshold_youden(yt, yp)

print("soglie di Youden (VAL):")
for nome in MODELLI_ATTIVI:
    print("  %-22s %.4f" % (nome, soglie[nome]))

#### Predizioni sul Test Set

In [ ]:
test_probs = {}
test_true = None
for nome, modello in modelli.items():
    yt, yp = predict_probs(modello, test_loader, DEVICE)
    test_true = yt
    test_probs[nome] = yp

def _key(nome):
    return nome.lower().replace('+', '_')

df_pred = df_test[['processed_path', 'cancer']].copy().rename(columns={'cancer': 'label_true'})
for nome in MODELLI_ATTIVI:
    k = _key(nome)
    df_pred['prob_' + k] = test_probs[nome]
    df_pred['pred_' + k] = (test_probs[nome] >= soglie[nome]).astype(int)

preds_path = os.path.join(PREDICTIONS_DIR, 'test_predictions.csv')
df_pred.to_csv(preds_path, index=False)
print("predizioni salvate in:", preds_path)

#### Valutazione — Metriche per Configurazione

In [ ]:
from sklearn.metrics import accuracy_score

COLORS = {'Real+Augmented': 'tab:orange', 'Real+SyntheticPositive': 'tab:green'}
CLASSIFICATORI = {nome: (test_probs[nome], soglie[nome]) for nome in MODELLI_ATTIVI}

all_metrics = {}
for nome, (probs, thr) in CLASSIFICATORI.items():
    fpr, tpr, _ = roc_curve(test_true, probs)
    y_pred = (probs >= thr).astype(int)
    all_metrics[nome] = {
        'auc': round(float(roc_auc_score(test_true, probs)), 4),
        'threshold': round(float(thr), 4),
        'accuracy': round(float(accuracy_score(test_true, y_pred)), 4),
        'precision': round(float(precision_score(test_true, y_pred, pos_label=1, zero_division=0)), 4),
        'recall': round(float(recall_score(test_true, y_pred, pos_label=1, zero_division=0)), 4),
        'f1': round(float(f1_score(test_true, y_pred, pos_label=1, zero_division=0)), 4),
        'fpr': fpr, 'tpr': tpr, 'y_pred': y_pred,
    }
    print("\n[%s] AUC=%.4f F1=%.4f recall=%.4f acc=%.4f" % (
        nome, all_metrics[nome]['auc'], all_metrics[nome]['f1'],
        all_metrics[nome]['recall'], all_metrics[nome]['accuracy']))
    print(classification_report(test_true, y_pred, target_names=['Sano', 'Malato']))

delta = {}
if 'Real+Augmented' in all_metrics and 'Real+SyntheticPositive' in all_metrics:
    delta = {k: round(all_metrics['Real+SyntheticPositive'][k] - all_metrics['Real+Augmented'][k], 4)
             for k in ('auc', 'f1', 'recall', 'accuracy')}
    print("\ndelta (Real+SyntheticPositive - Real+Augmented):", delta)

metrics_to_save = {
    'modelli_attivi': MODELLI_ATTIVI,
    'configs': {k: {m: v for m, v in vals.items() if m not in ('fpr', 'tpr', 'y_pred')}
                for k, vals in all_metrics.items()},
    'delta_synth_pos_minus_real_augmented': delta,
}
out_json = os.path.join(TABLES_DIR, 'test_metrics_trad_aug_vs_synth_pos_maxvit512.json')
with open(out_json, 'w') as f:
    json.dump(metrics_to_save, f, indent=2, ensure_ascii=False)
print("\nmetriche salvate in:", out_json)

#### ROC e Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for nome_conf, m in all_metrics.items():
    ax.plot(m['fpr'], m['tpr'], color=COLORS[nome_conf], lw=2,
            label=f"{nome_conf}  (AUC = {m['auc']:.4f})")

ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('ROC — Real+Augmented vs Real+SyntheticPositive (MaxViT-512, test reale)')
ax.legend(loc='lower right')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
roc_path = os.path.join(FIGURES_DIR, 'roc_trad_aug_vs_synth_pos_maxvit512.png')
fig.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvato in: {roc_path}")

In [ ]:
fig, axes = plt.subplots(1, len(all_metrics), figsize=(5 * len(all_metrics), 4), squeeze=False)
axes = axes[0]
for ax, (nome_conf, m) in zip(axes, all_metrics.items()):
    cm = confusion_matrix(test_true, m['y_pred'])
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['No cancro', 'Cancro'])
    ax.set_yticklabels(['No cancro', 'Cancro'])
    ax.set_xlabel('Predetto'); ax.set_ylabel('Reale')
    ax.set_title(f'{nome_conf}\nAUC={m["auc"]:.4f}   F1={m["f1"]:.4f}   Recall={m["recall"]:.4f}')
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=15, fontweight='bold', color=color)

plt.suptitle('Confusion Matrix — Test Set (MaxViT-512)', fontsize=13, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(FIGURES_DIR, 'cm_trad_aug_vs_synth_pos_maxvit512.png')
fig.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvato in: {cm_path}")

#### Confronto Metriche

In [ ]:
metric_keys   = ['auc', 'accuracy', 'precision', 'recall', 'f1']
metric_labels = ['AUC', 'Accuracy', 'Precision\n(cancer)', 'Recall\n(cancer)', 'F1\n(cancer)']

nomi = list(all_metrics.keys())
x = np.arange(len(metric_keys))
width = 0.8 / max(len(nomi), 1)

fig, ax = plt.subplots(figsize=(11, 5))
for i, nome_conf in enumerate(nomi):
    vals = [all_metrics[nome_conf][k] for k in metric_keys]
    bars = ax.bar(x + i * width, vals, width, label=nome_conf,
                  color=COLORS.get(nome_conf, 'tab:blue'), alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width * (len(nomi) - 1) / 2)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Confronto metriche — Real+Augmented vs Real+SyntheticPositive (MaxViT-512)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
bar_path = os.path.join(FIGURES_DIR, 'metrics_comparison_maxvit512.png')
fig.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvato in: {bar_path}")

#### Trade-off Sostenibilità

Riusa i log di sostenibilità (`codecarbon` + `psutil`, vedi [`eco_tracker.py`](eco_tracker.py)) prodotti da
`02_Data_Augmentation_Trad` (costo dell'augmentation) e da `03b`/`04b` (costo di training dei diffusori):
questi costi di **generazione dei dati** non cambiano al variare del classificatore a valle (ResNet-50 o
MaxViT-512), solo le prestazioni ottenute con quei dati cambiano.

In [ ]:
from pathlib import Path

ROOT = Path(BASE_PATH)
ECO_SOURCES = {
    'Augmentation': {'path': ROOT / 'results' / '02_data_augmentation' / 'ecotracker' / 'augmentation_ecotracker.json', 'color': 'tab:orange'},
    'Diffusore Fine-Tuned': {'path': ROOT / 'results' / '03b_finetuning_filtered' / 'ecotracker', 'color': 'tab:green'},
    'Diffusore From Scratch': {'path': ROOT / 'results' / '04b_ldm_keras_v2_extra1361' / 'ecotracker', 'color': 'tab:red'},
}
PERF_METRICS = ['auc', 'f1', 'recall']
ECO_KEYS = {
    'energy_kwh': ['energy_kwh', 'energy', 'kwh', 'energy_consumed_kwh', 'total_energy_kwh', 'energy_consumed'],
    'co2_kg': ['co2_kg', 'co2', 'co2eq', 'co2_emissions_kg', 'emissions_kg', 'emissions', 'carbon_kg'],
}

def _estrai(rec, chiavi):
    for k in chiavi:
        if k in rec:
            try:
                v = float(rec[k])
                if np.isfinite(v):
                    return v
            except (TypeError, ValueError):
                pass
    return None

def _record_da_file(percorso):
    try:
        txt = Path(percorso).read_text(encoding='utf-8').strip()
    except OSError:
        return []
    if not txt:
        return []
    try:
        obj = json.loads(txt)
        if isinstance(obj, dict):
            return [obj]
        if isinstance(obj, list):
            return [r for r in obj if isinstance(r, dict)]
    except json.JSONDecodeError:
        pass
    recs = []
    for riga in txt.splitlines():
        if not riga.strip():
            continue
        try:
            obj = json.loads(riga.strip())
            if isinstance(obj, dict):
                recs.append(obj)
        except json.JSONDecodeError:
            pass
    return recs

def leggi_eco(percorso):
    p = Path(percorso)
    if p.is_dir():
        files = sorted(f for f in p.iterdir() if f.suffix.lower() in ('.json', '.jsonl'))
    elif p.is_file():
        files = [p]
    else:
        return None
    e, c = 0.0, 0.0
    found = False
    for f in files:
        for rec in _record_da_file(f):
            ev = _estrai(rec, ECO_KEYS['energy_kwh'])
            cv = _estrai(rec, ECO_KEYS['co2_kg'])
            if ev is not None:
                e += ev; found = True
            if cv is not None:
                c += cv; found = True
    return {'energy_kwh': e, 'co2_kg': c} if found else None

eco = {}
for nome, cfg in ECO_SOURCES.items():
    dati = leggi_eco(cfg['path'])
    if dati:
        eco[nome] = dati
    else:
        print(f"log eco mancanti per '{nome}' -> sorgente saltata ({cfg['path']})")

log_sostenibilita = [n for n in ECO_SOURCES if n in eco]
print("riepilogo emissioni")
for n in log_sostenibilita:
    print(f"  {n:<24} energia={eco[n]['energy_kwh']:.3g} kWh   CO2={eco[n]['co2_kg']:.3g} kg")

In [ ]:
if log_sostenibilita and 'Real+Augmented' in all_metrics and 'Real+SyntheticPositive' in all_metrics:
    tot_co2 = sum(eco[n]['co2_kg'] for n in log_sostenibilita)
    tot_kwh = sum(eco[n]['energy_kwh'] for n in log_sostenibilita)
    quota = {n: {
        'co2': (eco[n]['co2_kg'] / tot_co2 * 100) if tot_co2 else float('nan'),
        'kwh': (eco[n]['energy_kwh'] / tot_kwh * 100) if tot_kwh else float('nan'),
    } for n in log_sostenibilita}

    pct_perf = {m: (all_metrics['Real+SyntheticPositive'][m] - all_metrics['Real+Augmented'][m])
                / all_metrics['Real+Augmented'][m] * 100 for m in PERF_METRICS}

    fig, (axc, axp) = plt.subplots(1, 2, figsize=(13, 5))
    xi = np.arange(2); w = 0.8 / len(log_sostenibilita)
    for i, nome in enumerate(log_sostenibilita):
        vals = [quota[nome]['co2'], quota[nome]['kwh']]
        axc.bar(xi + i * w, vals, w, color=ECO_SOURCES[nome]['color'], alpha=0.85, edgecolor='white', label=nome)
    axc.set_xticks(xi + w); axc.set_xticklabels(['CO2', 'kWh']); axc.set_ylim(0, 105)
    axc.set_ylabel('Quota % sul totale'); axc.set_title('Quota % inquinamento sul totale (generazione dati)')
    axc.legend(fontsize=7); axc.grid(True, axis='y', ls='--', alpha=0.5)

    xp = np.arange(len(PERF_METRICS)); wp = 0.38
    for i, nome in enumerate(MODELLI_ATTIVI):
        vals = [all_metrics[nome][m] for m in PERF_METRICS]
        axp.bar(xp + i * wp, vals, wp, color=COLORS.get(nome, 'tab:blue'), alpha=0.85, edgecolor='white', label=nome)
    axp.set_xticks(xp + wp / 2); axp.set_xticklabels([m.upper() for m in PERF_METRICS])
    axp.set_ylim(0, 1.15); axp.set_ylabel('Valore metrica (Test, MaxViT-512)')
    axp.set_title('Prestazioni — Real+Augmented vs Real+SyntheticPositive')
    axp.legend(fontsize=8); axp.grid(axis='y', ls='--', alpha=0.5)

    fig.suptitle('Trade-off Sostenibilità (dati) vs Prestazioni (MaxViT-512)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    tradeoff_path = os.path.join(FIGURES_DIR, 'eco_tradeoff_inquinamento_vs_prestazioni_maxvit512.png')
    fig.savefig(tradeoff_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Salvato in: {tradeoff_path}")
else:
    print("dati di sostenibilità insufficienti per il grafico di trade-off")